# Clusterização de Imagens Baseada na Extração de Features

Autor:  Viviane da Rosa Sommer

Atualizado: 02/04/2025

## Objetivo:
Este notebook tem como objetivo aplicar o algoritmo de clusterização (umap - histograma - 15 clusters) no dataset de imagens de mergulho, para avaliar como ele vai ditribuir imagens nunca vistas.

## Importações Necessárias

In [ ]:
import os
import numpy as np
import pandas as pd
import plotly.express as px
from joblib import load
from umap.umap_ import UMAP
from typing import List
from PIL import Image
import cv2
import base64
from io import BytesIO
import io
import matplotlib.pyplot as plt

## Declaração de Constantes e Modelos

In [ ]:
image_folder = '../Imagens de Mergulho - Modelo de Segmentação/Dataset-Segm-Mergulho/original_images'
model_path = 'trained_models/kmeans_histogram_umap_15_clusters.joblib'
results_path = "clustering_results.csv"
feature_method = "histogram"
reduction_method = "umap"
n_clusters = 15
search_dirs = ["dataset/images/train", "dataset/images/test"]

## Funções Necessárias

In [ ]:
def load_images_from_folder(folder_path: str) -> tuple[list[Image.Image], list[str]]:
    """
    Loads images from a folder.

    Args:
        folder_path (str): Path to the folder containing images.

    Returns:
        tuple[list[Image.Image], list[str]]: A tuple containing a list of images and their filenames.
    """
    images = []
    filenames = []
    for filename in os.listdir(folder_path):
        file_path = os.path.join(folder_path, filename)
        if os.path.isfile(file_path) and filename.lower().endswith(('.png', '.jpg', '.jpeg')):
            img = Image.open(file_path).convert("RGB")
            images.append(img)
            filenames.append(filename)
    return images, filenames


def image_to_base64(image: Image.Image) -> str:
    """
    Converts an image to a base64-encoded string.

    Args:
        image (Image.Image): Image to be converted.

    Returns:
        str: Base64-encoded string representation of the image.
    """
    buffered = io.BytesIO()
    image.save(buffered, format="PNG")
    return base64.b64encode(buffered.getvalue()).decode('utf-8')


def resize_image(image: Image.Image, size: tuple[int, int] = (100, 100)) -> Image.Image:
    """
    Resizes an image.

    Args:
        image (Image.Image): Image to be resized.
        size (tuple[int, int], optional): Desired size. Defaults to (100, 100).

    Returns:
        Image.Image: Resized image.
    """
    return image.resize(size)


def extract_histogram(image: Image.Image, bins: tuple[int, int, int] = (8, 8, 8)) -> np.ndarray:
    """
    Extracts a histogram from an image.

    Args:
        image (Image.Image): Image to extract histogram from.
        bins (tuple[int, int, int], optional): Number of bins for the histogram. Defaults to (8, 8, 8).

    Returns:
        np.ndarray: Flattened histogram.
    """
    histogram = cv2.calcHist([np.array(image)], [0, 1, 2], None, bins, [0, 256, 0, 256, 0, 256])
    histogram = cv2.normalize(histogram, histogram).flatten()
    return histogram


def plot_3d_clusters(features: np.ndarray, labels: np.ndarray) -> None:
    """
    Plots clusters in 3D without hover or additional information.

    Args:
        features (np.ndarray): Feature array (3D coordinates).
        labels (np.ndarray): Cluster labels.
    """
    fig = plt.figure(figsize=(10, 8))
    ax = fig.add_subplot(111, projection='3d')

    scatter = ax.scatter(
        features[:, 0],  # X-axis
        features[:, 1],  # Y-axis
        features[:, 2],  # Z-axis
        c=labels,        # Color by cluster
        cmap='viridis',  # Colormap
        s=20             # Point size
    )

    ax.set_title('3D Cluster Plot')
    ax.set_xlabel('Dimension 1')
    ax.set_ylabel('Dimension 2')
    ax.set_zlabel('Dimension 3')

    plt.colorbar(scatter, ax=ax, label='Cluster')
    plt.show()



def find_image(file_name: str, search_dirs: list[str]) -> str | None:
    """
    Finds the path of an image file in a list of directories.

    Args:
        file_name (str): Name of the image file to find.
        search_dirs (list[str]): List of directories to search in.

    Returns:
        str | None: Path to the image file or None if not found.
    """
    for directory in search_dirs:
        image_path = os.path.join(directory, file_name).replace("npy", "jpg")
        if os.path.exists(image_path):
            return image_path
    return None


def plot_similar_images_from_csv(
    results_path: str,
    reference_images: list[Image.Image],
    reference_clusters: list[int],
    feature_method: str,
    reduction_method: str,
    n_clusters: int,
    n_examples: int = 3,
    search_dirs: list[str] = None
) -> None:
    """
    Plots reference images along with similar images from a CSV file.

    Args:
        results_path (str): Path to the CSV file containing clustering results.
        reference_images (list[Image.Image]): List of reference images.
        reference_clusters (list[int]): Clusters of the reference images.
        feature_method (str): Feature extraction method.
        reduction_method (str): Dimensionality reduction method.
        n_clusters (int): Number of clusters used in KMeans.
        n_examples (int, optional): Number of similar images to plot. Defaults to 3.
        search_dirs (list[str], optional): List of directories to search for images. Defaults to None.
    """
    clustering_results = pd.read_csv(results_path)
    selected_row = clustering_results[
        (clustering_results["Feature Method"] == feature_method) &
        (clustering_results["Reduction Method"] == reduction_method) &
        (clustering_results["n_clusters"] == n_clusters)
    ]
    if selected_row.empty:
        return

    files = eval(selected_row.iloc[0]["files"])
    labels_str = selected_row.iloc[0]["labels"]
    labels = [int(label) for label in labels_str.strip("[]").split()]

    for ref_image, ref_cluster in zip(reference_images, reference_clusters):
        similar_files = [files[i] for i, label in enumerate(labels) if label == ref_cluster]
        selected_similar_files = similar_files[:n_examples]
        similar_images = []
        for similar_filename in selected_similar_files:
            similar_image_path = find_image(similar_filename, search_dirs)
            if similar_image_path:
                try:
                    similar_images.append(Image.open(similar_image_path))
                except Exception:
                    pass

        if not similar_images:
            continue

        plt.figure(figsize=(15, 5))
        plt.suptitle(f"Cluster {ref_cluster}: Reference and Similar Images", fontsize=16)

        plt.subplot(1, n_examples + 1, 1)
        plt.imshow(ref_image)
        plt.title("Reference", fontsize=10)
        plt.axis("off")

        for i, (similar_img, similar_filename) in enumerate(zip(similar_images, selected_similar_files), start=2):
            plt.subplot(1, n_examples + 1, i)
            plt.imshow(similar_img)
            plt.title(f"Similar", fontsize=10)
            plt.axis("off")

        plt.tight_layout()
        plt.show()


def plot_all_images_and_similars_matplotlib(images: list[Image.Image], filenames: list[str], labels: np.ndarray, n_examples: int = 3) -> None:
    """
    Plots all reference images and their similar images using Matplotlib.

    Args:
        images (list[Image.Image]): List of images used in clustering.
        filenames (list[str]): List of filenames of the images.
        labels (np.ndarray): Cluster labels predicted by KMeans.
        n_examples (int, optional): Number of similar images to plot. Defaults to 3.
    """
    unique_clusters = np.unique(labels)
    for cluster in unique_clusters:
        cluster_indices = [i for i, label in enumerate(labels) if label == cluster]
        reference_index = cluster_indices[0]
        reference_image = images[reference_index]
        reference_filename = filenames[reference_index]
        similar_indices = [i for i in cluster_indices if i != reference_index]
        selected_images = [images[i] for i in similar_indices[:n_examples]]
        selected_filenames = [filenames[i] for i in similar_indices[:n_examples]]

        plt.figure(figsize=(15, 5))
        plt.suptitle(f"Cluster {cluster}: Reference and Similar Images", fontsize=16)

        plt.subplot(1, n_examples + 1, 1)
        plt.imshow(reference_image)
        plt.title(f"Reference\n{reference_filename}", fontsize=10)
        plt.axis("off")

        for i, (sel_img, sel_filename) in enumerate(zip(selected_images, selected_filenames), start=2):
            plt.subplot(1, n_examples + 1, i)
            plt.imshow(sel_img)
            plt.title(f"Similar\n{sel_filename}", fontsize=10)
            plt.axis("off")

        plt.tight_layout()
        plt.show()


## Aplicando as imagens de mergulho no modelo de clusterização

In [ ]:
images, filenames = load_images_from_folder(image_folder)
histograms = np.array([extract_histogram(img) for img in images])
umap = UMAP(n_components=3, random_state=42)  
reduced_features = umap.fit_transform(histograms)
kmeans_model = load(model_path)
labels = kmeans_model.predict(reduced_features)

## Analisando cluster nas imagens de mergulho

In [ ]:
plot_all_images_and_similars_matplotlib(images, filenames, labels, n_examples=3)

## Analisando cluster das imagens de mergulho com as imagens de treino do modelo

In [ ]:
plot_similar_images_from_csv(
    results_path,
    images,
    labels,
    feature_method,
    reduction_method,
    n_clusters,
    n_examples=3,
    search_dirs=search_dirs
)

## Visualização 3D das imagens no espaço

In [ ]:
plot_3d_clusters(features=reduced_features, labels=labels)

In [ ]:
!jupyter nbconvert --to html Underwater_Image_Clustering.ipynb